In [ ]:
"""
Kroki:
- wczytuje wig20_d.csv
- oblicza log-zwroty (100 * log-close returns)
- wybiera ARMA(p,q) (p,q w zakresie 0..3) po AIC
- dla wybranego ARMA dopasowuje modele wariancji: GARCH(1,1), EGARCH(1,1), GJR-GARCH(1,1) każdy z rozkładami: 'normal', 't', 'skewt'
- wykonuje diagnostykę (LB test dla reszt i reszt^2)
- oblicza 1-krokowy prognoz: warunkowa średnia, warunkowa wariancja
- wyznacza 95% przedział ufności i sprawdza, czy ostatni (rzeczywisty) zwrot w nim leży
- oblicza P(next_return < 0) na podstawie dopasowanego rozkładu
- zapisuje podstawowe wyniki do pliku CSV
"""
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import itertools
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model

# Ustawienia
CSV_FILE = "wig20_d.csv"
DATE_COL = "Data"
CLOSE_COL = "Zamkniecie"

# Zakresy ARMA do przeszukania
P_MAX = 3
Q_MAX = 3

# Wczytanie danych
df = pd.read_csv(CSV_FILE, parse_dates=[DATE_COL], dayfirst=True)
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df[[DATE_COL, CLOSE_COL]].dropna()

# Log return w procentach
df['logret'] = 100.0 * np.log(df[CLOSE_COL].astype(float) / df[CLOSE_COL].astype(float).shift(1))
df = df.dropna().reset_index(drop=True)

# Odfiltrowanie ostatniej obserwacji jako rzeczywistej
train = df['logret'].iloc[:-1].copy()
real_next = float(df['logret'].iloc[-1])
train_index = df[DATE_COL].iloc[:-1]

# Wybór ARMA(p,q) wg AIC
best_aic = np.inf
best_order = (0, 0)
for p, q in itertools.product(range(P_MAX+1), range(Q_MAX+1)):
    try:
        m = ARIMA(train, order=(p, 0, q))
        res = m.fit()
        if res.aic < best_aic:
            best_aic = res.aic
            best_order = (p, q)
    except Exception:
        continue

p_best, q_best = best_order
print(f"Wybrany ARMA(p,q) wg AIC: p={p_best}, q={q_best}, AIC={best_aic:.2f}")

# Lista modeli wariancji i rozkładów
vol_models = {
    "GARCH": {"vol": "Garch", "o": 0},      # standardowy GARCH(1,1) -> p=1,o=0,q=1
    "EGARCH": {"vol": "EGARCH", "o": 0},    # EGARCH(1,1)
    "GJR-GARCH": {"vol": "Garch", "o": 1}   # GJR implementowane przez GARCH z o=1
}
dists = {"normal": "normal", "t": "t", "skewt": "skewt"}

results_summary = []

# funk. pomocnicze do diagnostyki
def ljung_box_p(series, lags=10):
    lb = acorr_ljungbox(series, lags=[lags], return_df=True)
    return float(lb['lb_pvalue'].iloc[0])

# Dopasowanie modeli i prognozy 1-step ahead
for vol_name, vol_spec in vol_models.items():
    for dist_name, dist_code in dists.items():
        try:
            am = arch_model(train,
                            mean='AR',
                            lags=p_best,
                            vol=vol_spec['vol'],
                            p=1, o=vol_spec['o'], q=1,
                            dist=dist_code)
            fit = am.fit(disp='off')

            # prognoza 1-step ahead
            fcast = fit.forecast(horizon=1, reindex=False)

            # warunkowa średnia i wariancja dla następnego kroku
            # fcast.mean może istnieć lub nie; jeśli nie — obliczemy z parametrów AR
            try:
                cond_mean = float(fcast.mean.iloc[-1, 0])
            except Exception:
                # jeśli brak mean w forecast, oblicz z ostatnich lagów i parametrów AR
                params = fit.params.to_dict()
                mu = params.get('mu', 0.0)
                ar_terms = [params.get(f'ar.L{i}', 0.0) for i in range(1, p_best+1)]
                last_vals = train.values[-p_best:] if p_best > 0 else []
                # last_vals has oldest...newest, zip must align ar.L1 with last value
                cond_mean = mu
                if p_best > 0:
                    rev_last = last_vals[::-1]
                    cond_mean += sum(a * v for a, v in zip(ar_terms, rev_last))

            cond_var = float(fcast.variance.iloc[-1, 0])
            cond_std = np.sqrt(cond_var)

            # pobierz parametry rozkładu z modelu (dla t/skewt będzie shape)
            dist = fit.model.distribution
            if dist.num_params == 0:
                dist_params = []
            else:
                # ostatnie dist.num_params parametrów w fit.params są parametrami rozkładu
                dist_params = fit.params[-dist.num_params:]

            # oblicz 95% przedział ufności parametrycznie:
            q_lower = dist.ppf(0.025, parameters=dist_params)
            q_upper = dist.ppf(0.975, parameters=dist_params)
            ci_lower = cond_mean + cond_std * q_lower
            ci_upper = cond_mean + cond_std * q_upper

            # prawdopodobieństwo, że next_return < 0
            z0 = (0.0 - cond_mean) / cond_std
            p_neg = dist.cdf(z0, parameters=dist_params)

            # diagnostyka reszt i kwadratów reszt
            std_resid = fit.std_resid  # znormalizowane reszty
            lb_resid_p = ljung_box_p(std_resid, lags=10)
            lb_sqresid_p = ljung_box_p(std_resid**2, lags=10)

            results_summary.append({
                "vol_model": vol_name,
                "dist": dist_name,
                "params": fit.params,
                "aic": fit.aic,
                "bic": fit.bic,
                "cond_mean": cond_mean,
                "cond_var": cond_var,
                "ci95_lower": ci_lower,
                "ci95_upper": ci_upper,
                "real_next": real_next,
                "in_CI": (ci_lower <= real_next <= ci_upper),
                "P(next<0)": float(p_neg),
                "LB_resid_p(10)": lb_resid_p,
                "LB_sqresid_p(10)": lb_sqresid_p,
                "fit_summary": str(fit.summary())
            })
            print(f"OK: {vol_name} + {dist_name} | AIC {fit.aic:.2f} | P(next<0)={p_neg:.4f} | In_CI={results_summary[-1]['in_CI']}")
        except Exception as e:
            print(f"Failed {vol_name} + {dist_name}: {e}")

# Zapis wyników (podsumowanie)
out = []
for r in results_summary:
    out.append({
        "vol_model": r['vol_model'],
        "dist": r['dist'],
        "aic": r['aic'],
        "bic": r['bic'],
        "cond_mean": r['cond_mean'],
        "cond_sd": np.sqrt(r['cond_var']),
        "ci95_lower": r['ci95_lower'],
        "ci95_upper": r['ci95_upper'],
        "real_next": r['real_next'],
        "in_CI": r['in_CI'],
        "P_next_negative": r['P(next<0)'],
        "LB_resid_p(10)": r['LB_resid_p(10)'],
        "LB_sqresid_p(10)": r['LB_sqresid_p(10)']
    })

if out:
    pd.DataFrame(out).to_csv("garch_model_summary.csv", index=False)
    print("Zapisano 'garch_model_summary.csv' z podsumowaniem.")
else:
    print("Brak wyników do zapisania (wszystkie próby nie powiodły się).")

if results_summary:
    best = min(results_summary, key=lambda x: x['aic'])
    print("\nNajlepszy model wg AIC:", best['vol_model'], best['dist'])
    # wypisz skrót fit_summary do pliku tekstowego:
    with open("best_model_summary.txt", "w", encoding="utf-8") as f:
        f.write(best['fit_summary'])
    print("Szczegółowe podsumowanie najlepszego modelu zapisano do 'best_model_summary.txt'.")
else:
    print("Brak dopasowanych modeli - nic do analizy.")


Wybrany ARMA(p,q) wg AIC: p=0, q=0, AIC=7833.32
OK: GARCH + normal | AIC 7442.76 | P(next<0)=0.4843 | In_CI=True
OK: GARCH + t | AIC 7374.89 | P(next<0)=0.4867 | In_CI=True
OK: GARCH + skewt | AIC 7376.89 | P(next<0)=0.4868 | In_CI=True
OK: EGARCH + normal | AIC 7445.67 | P(next<0)=0.4858 | In_CI=True
OK: EGARCH + t | AIC 7377.09 | P(next<0)=0.4872 | In_CI=True
OK: EGARCH + skewt | AIC 7379.08 | P(next<0)=0.4875 | In_CI=True
OK: GJR-GARCH + normal | AIC 7386.63 | P(next<0)=0.4960 | In_CI=True
OK: GJR-GARCH + t | AIC 7339.09 | P(next<0)=0.4959 | In_CI=True
OK: GJR-GARCH + skewt | AIC 7341.08 | P(next<0)=0.4958 | In_CI=True
Zapisano 'garch_model_summary.csv' z podsumowaniem.

Najlepszy model wg AIC: GJR-GARCH t
Szczegółowe podsumowanie najlepszego modelu zapisano do 'best_model_summary.txt'.
